# IBM Watson NLP Emotion Classification on Yelp Reviews

This notebook applies IBM Watson NLP's `emotion_aggregated-workflow_en_stock` model to the `text` column of `yelp_reviews_clean.csv` stored in Google Drive.

It generates:
- emotion probabilities for each review: anger, disgust, fear, joy, and sadness
- an `emotion_breakdown` dictionary column
- a `dominant_emotion` column
- histograms for each emotion distribution
- average emotion distribution across all reviews
- the emotion with the highest average score


## Cell 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## Cell 2: Install / import required packages

In [ ]:
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
import os
import gc
from tqdm.auto import tqdm

In [ ]:
# If Watson NLP is not already installed in your environment, run this cell.
!pip install ibm-watson-nlp

In [ ]:
import watson_nlp

## Cell 3: Set file paths

In [ ]:
input_path = "/content/drive/MyDrive/yelp_reviews_clean.csv"

output_path = "/content/drive/MyDrive/yelp_reviews_with_emotions.csv"

summary_output_path = "/content/drive/MyDrive/yelp_emotion_summary.csv"

histogram_folder = "/content/drive/MyDrive/yelp_emotion_histograms"

os.makedirs(histogram_folder, exist_ok=True)

## Cell 4: Load IBM Watson emotion model

In [ ]:
# Load IBM Watson NLP emotion workflow model for English
emotion_model = watson_nlp.load(
    "emotion_aggregated-workflow_en_stock"
)

## Cell 5: Test the model on one sentence

In [ ]:
test_text = "I'm so annoyed that this code won't run."

emotion_result = emotion_model.run(test_text)

print(emotion_result)

The output should contain an `emotion_predictions` field with probabilities for anger, disgust, fear, joy, and sadness.

## Cell 6: Helper function to extract emotion scores

In [ ]:
def get_emotion_scores(text):
    """
    Runs IBM Watson NLP emotion classification on one review
    and returns a dictionary of emotion probabilities.
    """
    try:
        if pd.isna(text) or str(text).strip() == "":
            return {
                "anger": np.nan,
                "disgust": np.nan,
                "fear": np.nan,
                "joy": np.nan,
                "sadness": np.nan
            }

        result = emotion_model.run(str(text))

        # Watson NLP result usually contains emotion_predictions
        emotion_dict = result["emotion_predictions"][0]["emotion"]

        return {
            "anger": emotion_dict.get("anger", np.nan),
            "disgust": emotion_dict.get("disgust", np.nan),
            "fear": emotion_dict.get("fear", np.nan),
            "joy": emotion_dict.get("joy", np.nan),
            "sadness": emotion_dict.get("sadness", np.nan)
        }

    except Exception as error:
        return {
            "anger": np.nan,
            "disgust": np.nan,
            "fear": np.nan,
            "joy": np.nan,
            "sadness": np.nan
        }

## Cell 7: Process the Yelp reviews in chunks

This is the safer version for a large dataset. It avoids loading all reviews into memory.

In [ ]:
emotion_columns = [
    "anger",
    "disgust",
    "fear",
    "joy",
    "sadness"
]

chunk_size = 5_000
first_chunk = True
total_rows_processed = 0

# Delete previous output file if it exists
if os.path.exists(output_path):
    os.remove(output_path)

reader = pd.read_csv(
    input_path,
    chunksize=chunk_size
)

for chunk_number, chunk in enumerate(reader, start=1):

    print(f"Processing chunk {chunk_number}...")

    emotion_results = []

    for review_text in tqdm(chunk["text"], desc=f"Chunk {chunk_number}"):
        scores = get_emotion_scores(review_text)
        emotion_results.append(scores)

    emotion_df = pd.DataFrame(emotion_results)

    # Add separate emotion probability columns
    for col in emotion_columns:
        chunk[col] = emotion_df[col].values

    # Store full dictionary as a JSON-style string
    chunk["emotion_breakdown"] = emotion_df.apply(
        lambda row: {
            "anger": row["anger"],
            "disgust": row["disgust"],
            "fear": row["fear"],
            "joy": row["joy"],
            "sadness": row["sadness"]
        },
        axis=1
    )

    chunk["emotion_breakdown"] = chunk["emotion_breakdown"].apply(
        json.dumps
    )

    # Store dominant emotion for each review
    chunk["dominant_emotion"] = chunk[emotion_columns].idxmax(axis=1)

    # Save chunk to output CSV
    chunk.to_csv(
        output_path,
        mode="w" if first_chunk else "a",
        header=first_chunk,
        index=False,
        encoding="utf-8"
    )

    first_chunk = False
    total_rows_processed += len(chunk)

    print(
        f"Finished chunk {chunk_number}; "
        f"total rows processed: {total_rows_processed:,}"
    )

    del chunk, emotion_df, emotion_results
    gc.collect()

print("Emotion classification completed.")
print("Output saved to:", output_path)

## Cell 8: Read emotion columns only for analysis

In [ ]:
emotion_data = pd.read_csv(
    output_path,
    usecols=emotion_columns + ["dominant_emotion"]
)

print(emotion_data.head())

## Cell 9: Compute average emotion distribution

In [ ]:
average_emotions = emotion_data[emotion_columns].mean().sort_values(
    ascending=False
)

print("Average emotion distribution:")
print(average_emotions)

max_average_emotion = average_emotions.idxmax()
max_average_value = average_emotions.max()

print()
print("Emotion with highest average score:", max_average_emotion)
print("Average score:", max_average_value)

## Cell 10: Save average emotion summary

In [ ]:
summary_df = average_emotions.reset_index()
summary_df.columns = ["emotion", "average_score"]

summary_df.to_csv(
    summary_output_path,
    index=False
)

print("Emotion summary saved to:", summary_output_path)

display(summary_df)

## Cell 11: Plot average emotion distribution

In [ ]:
plt.figure(figsize=(8, 5))

plt.bar(
    summary_df["emotion"],
    summary_df["average_score"]
)

plt.title("Average Emotion Distribution Across Yelp Reviews")
plt.xlabel("Emotion")
plt.ylabel("Average Probability")
plt.ylim(0, 1)
plt.xticks(rotation=45)
plt.tight_layout()

average_plot_path = os.path.join(
    histogram_folder,
    "average_emotion_distribution.png"
)

plt.savefig(average_plot_path, dpi=300)
plt.show()

print("Saved plot to:", average_plot_path)

## Cell 12: Generate histograms for each emotion

In [ ]:
for emotion in emotion_columns:
    plt.figure(figsize=(8, 5))

    plt.hist(
        emotion_data[emotion].dropna(),
        bins=50
    )

    plt.title(f"Distribution of {emotion.capitalize()} Scores")
    plt.xlabel(f"{emotion.capitalize()} Probability")
    plt.ylabel("Number of Reviews")
    plt.xlim(0, 1)
    plt.tight_layout()

    plot_path = os.path.join(
        histogram_folder,
        f"{emotion}_distribution.png"
    )

    plt.savefig(plot_path, dpi=300)
    plt.show()

    print("Saved:", plot_path)

## Optional: Run on a small sample first

Running IBM’s emotion model on millions of reviews may take a long time. Start with a smaller sample first if needed.

In [ ]:
sample_df = pd.read_csv(input_path, nrows=1000)

sample_df["emotion_scores"] = sample_df["text"].apply(
    get_emotion_scores
)

display(sample_df[["text", "emotion_scores"]].head())